# Partie B — PLM utilisé directement (aucun entraînement)

Deux modèles déjà entraînés ailleurs. Aucun gradient sur ArSarcasm-v2.

La validation reste indispensable : pour le zero-shot, `P(sarcastique)` est seuillée sur `val` par le F1 sarcastique. Le fichier de test officiel n'est pas scoré.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from data import load_and_split
from evaluate import append_results, build_part_b_rows, sarcasm_f1
from models.plm import (
    HYPOTHESES,
    NLI_MODEL,
    SENTIMENT_MODEL,
    collect_part_b,
    hypothesis_frame,
    load_nli_pipeline,
    load_sentiment_pipeline,
    slice_validation,
)
from train import DEFAULT_SEED, set_seed

pd.set_option("display.max_colwidth", 160)
set_seed(DEFAULT_SEED)

LIMIT = None  # entier : essai court, results.csv n'est pas modifié
BATCH_SIZE = 8

bundle = load_and_split()
val = slice_validation(bundle["val"], LIMIT)
y_val = val["sarcasm"].astype(bool)
print(f"validation scorée : {len(val)} tweets")
print(f"NLI        : {NLI_MODEL}")
print(f"sentiment  : {SENTIMENT_MODEL}")

validation scorée : 1255 tweets
NLI        : MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7
sentiment  : CAMeL-Lab/bert-base-arabic-camelbert-da-sentiment


## B1. Zero-shot par NLI

Le tweet est la prémisse. Le gabarit remplace `{}` par une étiquette et devient l'hypothèse. Le pipeline `zero-shot-classification` renvoie les étiquettes **triées par score** : on relit le score de l'étiquette sarcastique, pas `scores[0]`.

Trois formulations : arabe, français, anglais. Pour chacune, le seuil de `P(sarcastique)` est choisi sur `val` (grille 0.01–0.99, F1 sarcastique maximal). Aucun paramètre n'est entraîné.

In [2]:
display(hypothesis_frame())
for hypothesis in HYPOTHESES:
    print(hypothesis.name, "→", hypothesis.template.format(hypothesis.positive))

,model,langue,gabarit,etiquettes,positive,hypothese
0,nli_zeroshot_ar,ar,هذه التغريدة {}.,ساخرة / جادة,ساخرة,هذه التغريدة ساخرة.
1,nli_zeroshot_fr,fr,Ce tweet est {}.,sarcastique / sérieux,sarcastique,Ce tweet est sarcastique.
2,nli_zeroshot_en,en,This tweet is {}.,sarcastic / serious,sarcastic,This tweet is sarcastic.


nli_zeroshot_ar → هذه التغريدة ساخرة.
nli_zeroshot_fr → Ce tweet est sarcastique.
nli_zeroshot_en → This tweet is sarcastic.


## B2. Transfert par heuristique

`CAMeL-Lab/bert-base-arabic-camelbert-da-sentiment` est déjà fine-tuné. La règle ne l'entraîne pas : **sentiment prédit négatif ⇒ sarcastique**.

La classe négative est lue dans `model.config.id2label` par son nom, pas par un indice figé. La décision est l'argmax du modèle. `P(sarcastique)` pour le journal est `P(négatif)`, ce qui permet le PR-AUC sans inventer un second seuil.

In [3]:
nli = load_nli_pipeline()
sentiment = load_sentiment_pipeline()
print(sentiment.model.config.id2label)

report = collect_part_b(val, nli, sentiment, batch_size=BATCH_SIZE)

for run in report.nli:
    f1 = sarcasm_f1(y_val, run.pred)
    print(f"{run.name} — seuil {run.threshold:.2f} — F1 val {f1:.4f} — {run.elapsed_s:.1f} s")

sentiment_f1 = sarcasm_f1(y_val, report.sentiment_pred)
print(
    f"sentiment_neg_implies_sarcasm — argmax — F1 val {sentiment_f1:.4f} — "
    f"{report.sentiment_elapsed_s:.1f} s"
)
print("id2label retenu :", report.id2label)

config.json:   0%|          | 0.00/1.09k [00:00<?, ?B/s]

c:\ghoufrane\Master\M2\llms\TP2\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--MoritzLaurer--mDeBERTa-v3-base-xnli-multilingual-nli-2mil7. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\ghoufrane\Master\M2\llms\TP2\.venv\Lib\site-packages\torch\jit\_script.

model.safetensors: reconstructing file:   0%|          |  0.00B /  558MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

...

model.safetensors: downloading bytes:           |  0.00B            

vocab.txt:   0%|          | 0.00/305k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

{0: 'positive', 1: 'negative', 2: 'neutral'}


KeyboardInterrupt: 

## Piège — contamination

Le modèle de B2 a été fine-tuné sur ASTD, ArSAS et SemEval. Or ArSarcasm (v1), inclus dans ArSarcasm-v2, a été construit en ré-annotant ASTD et SemEval 2017.

Il a donc peut-être déjà vu certains tweets. On ne retire rien : la note reste dans le journal, et le score de B2 se lit avec cette réserve. Le test officiel n'est toujours pas utilisé.

In [ ]:
rows = build_part_b_rows(
    y_val,
    [(run.name, run.pred, run.proba, run.threshold, run.elapsed_s) for run in report.nli],
    report.sentiment_pred,
    report.sentiment_proba,
    report.sentiment_elapsed_s,
    seed=DEFAULT_SEED,
)
display(pd.DataFrame(rows))

if LIMIT is None:
    saved = append_results(rows, ROOT / "results" / "results.csv")
    display(saved)
    print("Prédictions de test non écrites : elles sont réservées à la partie F.")
else:
    print("LIMIT est actif : results.csv n'est pas modifié.")